# Semana 3 - Analisis Tecnico

Probamos RSI, MACD, Bandas de Bollinger y soportes/resistencias.
Foco: YPF (papel principal del portfolio) y comparacion con el resto.

In [ ]:
import sys
sys.path.append(".")

from src.tecnico import (
    rsi, macd, bollinger,
    soportes_resistencias,
    señales_tecnicas, mostrar_señales,
)
from src.graficos import (
    graficar_rsi, graficar_macd,
    graficar_bollinger, graficar_soportes_resistencias,
    grafico_candlestick,
)
from src.datos import cargar_universo

universo = cargar_universo()
print("Modulos tecnicos cargados")

## 1. Resumen de senales para YPF

In [ ]:
mostrar_señales("YPF", periodo="1y")

## 2. RSI - Indice de Fuerza Relativa

**Lectura rapida**:
- RSI > 70: sobrecompra, papel "caro" en lo inmediato
- RSI < 30: sobreventa, papel "barato" en lo inmediato
- RSI entre 30-70: zona neutral

In [ ]:
graficar_rsi("YPF", periodo="1y", ventana=14)

In [ ]:
rsi_serie = rsi("YPF", ventana=14, periodo="1y")
rsi_serie.tail(10).round(2)

## 3. MACD - Momentum y cruces

**Lectura rapida**:
- MACD cruza Senal de abajo hacia arriba = senal de compra
- MACD cruza Senal de arriba hacia abajo = senal de venta
- Histograma creciente = momentum acelerando

In [ ]:
graficar_macd("YPF", periodo="1y")

In [ ]:
macd_df = macd("YPF", periodo="1y")
macd_df.tail(10).round(3)

## 4. Bandas de Bollinger - Volatilidad y mean-reversion

**Lectura rapida**:
- Precio toca banda superior = posible techo de corto plazo
- Precio toca banda inferior = posible piso de corto plazo
- Bandas estrechas = baja volatilidad, suele preceder movimientos fuertes

In [ ]:
graficar_bollinger("YPF", periodo="1y", ventana=20, desv=2)

In [ ]:
bb = bollinger("YPF", periodo="1y")
bb.tail(5).round(2)

## 5. Soportes y Resistencias

Detecta niveles donde el precio rebotó historicamente (max/min locales).
Util para definir stops y targets.

In [ ]:
sr = soportes_resistencias("YPF", periodo="6mo", ventana=10, top_n=5)
print(f"Precio actual: ${sr['precio_actual']:.2f}\n")

print("RESISTENCIAS (techos arriba del precio):")
for fecha, nivel in sr["resistencias"]:
    print(f"  ${nivel:.2f}  ({fecha.strftime('%d %b %Y')})")

print("\nSOPORTES (pisos abajo del precio):")
for fecha, nivel in sr["soportes"]:
    print(f"  ${nivel:.2f}  ({fecha.strftime('%d %b %Y')})")

In [ ]:
graficar_soportes_resistencias("YPF", periodo="6mo", ventana=10, top_n=5)

## 6. Comparacion: senales del portfolio completo

In [ ]:
import pandas as pd

mi_portfolio = universo["portfolios"]["personal"]
filas = []
for t in mi_portfolio:
    s = señales_tecnicas(t, periodo="1y")
    if s:
        filas.append({
            "Ticker": s["ticker"],
            "Precio": s["precio_actual"],
            "RSI": s["RSI"],
            "RSI lectura": s["RSI_interpretacion"],
            "MACD lectura": s["MACD_interpretacion"],
            "BB posicion %": s["BB_posicion_%"],
        })

df_senales = pd.DataFrame(filas)
df_senales

## 7. Cross-check: candlestick puro de YPF

Para comparar lo que dicen los indicadores contra el grafico real.

In [ ]:
grafico_candlestick("YPF", periodo="6mo")